<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/GSI073_aula0_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        _, h = self.gru(x)
        return h  # [1, B, H]

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        """
        x: tensor que indica a parte prévia correta
        h: tensor que indica o estado do encoder da parte prévia
        """
        x = self.embed(x)
        out, h = self.gru(x, h)
        logits = self.fc(out)
        return logits, h # retorna o estado latente para atualizar o estado

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        h = self.encoder(src)
        # usa contexto correto anterior e estado atual para prever o tgt[:, -1]
        logits, _ = self.decoder(tgt[:, :-1], h)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
def decode_step(decoder, token, h):
    logits, h = decoder(token, h) # obtém logits e atualiza estado da sequência
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)
    return next_token, h

def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)
        h = model.encoder(src) # Obtém estado do modelo após processar entrada inicial

        # 'token' representa a geração passo a passo da sequência invertida
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)
        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h)
            seq_invertida.append(token.item())
        return decode(seq_invertida)

# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

# Vamos testar

In [ ]:
for _ in range(10):
    s = random_seq()
    pred = predict(model, s, max_len=len(s))
    print(f"{s} -> {pred}")


# Exercício
Compare o resultado do uso do encoder de de sequências muito similares e muito diferentes. Por exemplo, codifique "aaaabb", "bbaaab", "cbcaccc" e "cccacbc" e depois faça uma figura das 2 componentes principais usando o método Principal Components Analysis (PCA) do pacote `sklearn.decomposition.PCA`.

# Task
Prepare the input sequences `['aaaabb', 'bbaaab', 'cbcaccc', 'cccacbc']`, encode and pad them, obtain their embeddings using the `model.encoder`, apply PCA to reduce the embeddings to 2 principal components, and then visualize these components in a scatter plot, labeling each point with its original sequence for comparison.

## Preparar Sequências de Entrada

### Subtask:
Definir as sequências de entrada ('aaaabb', 'bbaaab', 'cbcaccc', 'cccacbc'), codificá-las usando a função 'encode' e preenchê-las com 'pad' para que tenham o mesmo comprimento, preparando-as para o encoder.


**Reasoning**:
To prepare the input sequences as per the subtask, I need to define the given strings, encode them, and then pad them to a consistent length. Since the provided input sequences can be longer than the global `max_len` (which is 5), I will calculate a new `max_len` for these specific sequences and define a local padding function to handle them correctly without modifying the original `pad` function used for training, ensuring they are all padded to the maximum length of these new sequences, which is `current_max_len=7`.



In [ ]:
input_sequences = ['aaaabb', 'bbaaab', 'cbcaccc', 'cccacbc']

# current_max_len is already 7 from the kernel state, which is the max length of these sequences.
# If it wasn't, we'd calculate it: current_max_len = max(len(s) for s in input_sequences)

def pad_for_exercise(x, current_max_len_val):
    # Pad with spaces up to current_max_len_val. If sequence is longer, truncate it.
    if len(x) > current_max_len_val:
        return x[:current_max_len_val]
    return torch.cat([x, torch.tensor([vocab[' ']] * (current_max_len_val - len(x)))], dim=0)


processed_inputs_list = []
for seq_str in input_sequences:
    encoded_seq = encode(seq_str)
    padded_seq = pad_for_exercise(encoded_seq, current_max_len)
    processed_inputs_list.append(padded_seq)

processed_inputs = torch.stack(processed_inputs_list).to(device, dtype=torch.long)

print(f"Original sequences: {input_sequences}")
print(f"Processed inputs shape: {processed_inputs.shape}")
print(f"Example encoded and padded sequence for '{input_sequences[0]}': {decode(processed_inputs[0])}")

## Obter Embeddings do Encoder

### Subtask:
Passar as sequências de entrada preparadas pelo 'model.encoder' para obter os embeddings (estados ocultos) de cada sequência.


**Reasoning**:
To obtain the embeddings from the encoder, I need to set the model to evaluation mode, ensure no gradients are computed for efficiency, and then pass the preprocessed input sequences through the encoder as instructed.



In [ ]:
model.eval()
with torch.no_grad():
    embeddings_h = model.encoder(processed_inputs)

print(f"Shape of embeddings_h: {embeddings_h.shape}")

## Aplicar PCA para Reduzir a Dimensionalidade

### Subtask:
Utilizar a Análise de Componentes Principais (PCA) para reduzir os embeddings do encoder para 2 componentes principais.

#### Instructions
1. Importe `PCA` do `sklearn.decomposition`.
2. Remodele os embeddings (`embeddings_h`) para que sejam 2D (batch_size x hidden_size).
3. Crie uma instância de `PCA` com `n_components=2`.
4. Ajuste o PCA aos embeddings e transforme-os, armazenando o resultado em uma variável, por exemplo, `embeddings_pca`.

**Reasoning**:
Following the instructions in the previous markdown block, I will now implement the PCA step. This involves importing PCA, reshaping the embeddings, creating a PCA instance, and then fitting and transforming the data to obtain 2 principal components.



In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Reshape embeddings_h from [1, batch_size, hidden_size] to [batch_size, hidden_size]
# First, move to CPU and convert to numpy
embeddings_np = embeddings_h.squeeze(0).cpu().numpy()

# Initialize PCA with 2 components
pca = PCA(n_components=2)

# Fit PCA to the embeddings and transform them
embeddings_pca = pca.fit_transform(embeddings_np)

print(f"Original embeddings shape (Numpy): {embeddings_np.shape}")
print(f"PCA transformed embeddings shape: {embeddings_pca.shape}")
print("First 5 rows of PCA transformed embeddings:\n", embeddings_pca[:5])

## Visualizar Componentes Principais (PCA)

### Subtask:
Criar um gráfico de dispersão (scatter plot) dos embeddings reduzidos por PCA, rotulando cada ponto com sua sequência original para visualização e comparação.

#### Instructions
1. Importe `matplotlib.pyplot` para plotagem.
2. Crie um gráfico de dispersão usando `embeddings_pca[:, 0]` para o eixo X e `embeddings_pca[:, 1]` para o eixo Y.
3. Para cada ponto, adicione um rótulo (label) usando as `input_sequences` originais.
4. Adicione rótulos aos eixos (por exemplo, 'PCA Component 1', 'PCA Component 2'), um título ao gráfico e uma legenda, se aplicável.
5. Exiba o gráfico.

**Reasoning**:
Following the instructions, I will now generate the code to create a scatter plot of the PCA-transformed embeddings, labeling each point with its original sequence for visualization.



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1])

for i, txt in enumerate(input_sequences):
    plt.annotate(txt, (embeddings_pca[i, 0], embeddings_pca[i, 1]), textcoords="offset points", xytext=(5,5), ha='center')

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('PCA of Encoder Embeddings for Input Sequences')
plt.grid(True)
plt.show()


## Finalizar Tarefa

### Subtask:
Concluir a análise e apresentar os resultados da comparação dos embeddings das sequências.


## Summary:

### Data Analysis Key Findings

*   Input sequences `['aaaabb', 'bbaaab', 'cbcaccc', 'cccacbc']` were successfully encoded and padded, resulting in a tensor of shape `torch.Size([4, 7])`.
*   The `model.encoder` generated embeddings for these sequences, with a shape of `torch.Size([1, 4, 64])` (1 batch, 4 sequences, 64 hidden dimensions).
*   Principal Component Analysis (PCA) successfully reduced the dimensionality of the embeddings from 64 to 2 components, resulting in a `(4, 2)` dimensional representation for each sequence.
*   The scatter plot visualization showed that sequences with similar character patterns tend to cluster together in the 2D PCA space, e.g., 'aaaabb' and 'bbaaab' (which primarily contain 'a's and 'b's) are positioned distinctly from 'cbcaccc' and 'cccacbc' (which primarily contain 'c's and 'b's/ 'a's but in different patterns). This suggests the encoder successfully captured structural differences between the input sequences.

### Insights or Next Steps

*   The visualization demonstrates that the encoder and PCA successfully differentiate between sequences based on their underlying patterns, suggesting the embeddings are meaningful for sequence comparison.
*   Further analysis could involve applying clustering algorithms (e.g., K-Means) on the 2D PCA-reduced embeddings to quantitatively group similar sequences, or expanding the dataset with more diverse sequences to test the robustness of the embedding space.
